In [1]:
import ctypes
import csv
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import chipwhisperer as cw
from picosdk.ps6000a import ps6000a as ps
from picosdk.PicoDeviceEnums import picoEnum as enums
from picosdk.functions import adc2mV, mV2adc, assert_pico_ok

C:\Users\gouth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CW305 target: update these two Windows paths if the files move.
BITSTREAM_PATH = r"C:\Users\gouth\Desktop\AES_withPicoScope\AES_withPicoScope\AES_verilogFiles\cw305_top.bit"
DEFINES_PATH = r"C:\Users\gouth\Desktop\AES_withPicoScope\AES_withPicoScope\AES_verilogFiles\cw305_aes_defines.v"
FPGA_ID = "100t"

# Number of captures and samples per waveform.
N_TRACES = 20_000          # total: 10,000 random and 10,000 fixed
N_SAMPLES = 5_000
PRE_TRIGGER = 1_000
POST_TRIGGER = N_SAMPLES - PRE_TRIGGER

OUT_RANDOM = "custom_aes_traces_random.csv"
OUT_FIXED  = "custom_aes_traces_fixed.csv"
OVERWRITE_OUTPUT = True    # set False to avoid replacing existing datasets

# PicoScope connections:
# CW305 X4       -> PicoScope Channel A (measured waveform)
# CW305 TP1/TRIG -> PicoScope Channel C (trigger, P2056 10x probe)
# P2056 ground  -> CW305 GND4

CHANNEL_A_RANGE = 2  #Channel A to the ±50 mV
CHANNEL_C_RANGE = 5  #Channel C to the ±500 mV
TRIGGER_THRESHOLD_MV = 150  # external 3.3 V trigger is ~330 mV after 10x probe
AUTO_TRIGGER_US = 0 # Zero disables the automatic trigger timeout
TIMEOUT_S = 5.0 # Python waits at most 5 seconds for acquisition completion
RESOLUTION = enums.PICO_DEVICE_RESOLUTION["PICO_DR_12BIT"]

# Known AES-128 test key and fixed plaintext.
FIXED_KEY = bytearray.fromhex("000102030405060708090a0b0c0d0e0f")
FIXED_PLAINTEXT = bytearray.fromhex("00112233445566778899aabbccddeeff")

assert N_TRACES >= 2 and N_TRACES % 2 == 0, "N_TRACES must be even."
assert N_SAMPLES > PRE_TRIGGER >= 0  # pre-trigger sample count is nonnegative and smaller than the total sample count.

In [ ]:
class Pico6000EBlock:
    """Small wrapper around the PicoScope 6000E block-capture API.

    The detailed SDK calls are hardware-specific; beginners can use the
    five methods below without changing their implementation.
    """
    def __init__(self):
        self.chandle = ctypes.c_int16()
        self.status = {}

        self.min_adc = ctypes.c_int16()
        self.max_adc = ctypes.c_int16()

        self.timebase = ctypes.c_uint32(0)
        self.time_interval_s = ctypes.c_double(0)

        self.buffer_a_max = None
        self.buffer_a_min = None

        self.downsample_mode = enums.PICO_RATIO_MODE["PICO_RATIO_MODE_RAW"]

    def open(self):
        """Connect to the scope and read its ADC limits."""
        self.status["openunit"] = ps.ps6000aOpenUnit(
            ctypes.byref(self.chandle),
            None,
            RESOLUTION
        )
        assert_pico_ok(self.status["openunit"])

        self.status["getAdcLimits"] = ps.ps6000aGetAdcLimits(
            self.chandle,
            RESOLUTION,
            ctypes.byref(self.min_adc),
            ctypes.byref(self.max_adc)
        )
        assert_pico_ok(self.status["getAdcLimits"])

        print("PicoScope opened.")
        print("Max ADC:", self.max_adc.value)

    def configure(self):
        """Set measurement/trigger channels, timebase, and sample buffers."""
        ch_a = enums.PICO_CHANNEL["PICO_CHANNEL_A"]
        ch_b = enums.PICO_CHANNEL["PICO_CHANNEL_B"]
        ch_c = enums.PICO_CHANNEL["PICO_CHANNEL_C"]
        ch_d = enums.PICO_CHANNEL["PICO_CHANNEL_D"]

        dc_50 = enums.PICO_COUPLING["PICO_DC_50OHM"]
        dc_1m = enums.PICO_COUPLING["PICO_DC"]

        # BEST METRICS: 20 MHz bandwidth limit to remove high-frequency noise 
        # and improve CPA/TVLA Signal-to-Noise Ratio (SNR)
        bw_limit = enums.PICO_BANDWIDTH_LIMITER["PICO_BW_20MHZ"]
        bw_full = enums.PICO_BANDWIDTH_LIMITER["PICO_BW_FULL"]

        # Channel A: capture trace from CW305 X4.
        self.status["setChannelA"] = ps.ps6000aSetChannelOn(
            self.chandle,
            ch_a,
            dc_50,
            CHANNEL_A_RANGE,
            0,
            bw_full  # Applied the 20 MHz limit here
        )
        assert_pico_ok(self.status["setChannelA"])

        # Channel C: trigger from CW305 TP1/TRIG using P2056 10x probe.
        self.status["setChannelC"] = ps.ps6000aSetChannelOn(
            self.chandle,
            ch_c,
            dc_1m,
            CHANNEL_C_RANGE,
            0,
            bw_full
        )
        assert_pico_ok(self.status["setChannelC"])

        # Disable unused channels to free up ADC resources.
        for ch_name, ch in [("B", ch_b), ("D", ch_d)]:
            self.status[f"setChannel{ch_name}Off"] = ps.ps6000aSetChannelOff(
                self.chandle,
                ch
            )
            assert_pico_ok(self.status[f"setChannel{ch_name}Off"])

        # Trigger on Channel C rising edge.
        threshold_adc = mV2adc(
            TRIGGER_THRESHOLD_MV,
            CHANNEL_C_RANGE,
            self.max_adc
        )

        self.status["setSimpleTrigger"] = ps.ps6000aSetSimpleTrigger(
            self.chandle,
            1,  # enable
            ch_c,
            threshold_adc,
            enums.PICO_THRESHOLD_DIRECTION["PICO_RISING"],
            0,  # trigger delay
            AUTO_TRIGGER_US
        )
        assert_pico_ok(self.status["setSimpleTrigger"])

        # Get fastest available valid timebase for channels A and C.
        enabled_flags = (
            enums.PICO_CHANNEL_FLAGS["PICO_CHANNEL_A_FLAGS"]
            | enums.PICO_CHANNEL_FLAGS["PICO_CHANNEL_C_FLAGS"]
        )

        self.status["getMinimumTimebaseStateless"] = ps.ps6000aGetMinimumTimebaseStateless(
            self.chandle,
            enabled_flags,
            ctypes.byref(self.timebase),
            ctypes.byref(self.time_interval_s),
            RESOLUTION
        )
        assert_pico_ok(self.status["getMinimumTimebaseStateless"])

        print("Pico timebase:", self.timebase.value)
        print("Sample interval:", self.time_interval_s.value, "s")

        # Allocate Channel A buffers.
        self.buffer_a_max = (ctypes.c_int16 * N_SAMPLES)()
        self.buffer_a_min = (ctypes.c_int16 * N_SAMPLES)()

        self.status["setDataBuffersA"] = ps.ps6000aSetDataBuffers(
            self.chandle,
            ch_a,
            ctypes.byref(self.buffer_a_max),
            ctypes.byref(self.buffer_a_min),
            N_SAMPLES,
            enums.PICO_DATA_TYPE["PICO_INT16_T"],
            0,  # segment index
            self.downsample_mode,
            enums.PICO_ACTION["PICO_CLEAR_ALL"] | enums.PICO_ACTION["PICO_ADD"]
        )
        assert_pico_ok(self.status["setDataBuffersA"])

        print("PicoScope configured: Channel A trace, Channel C trigger.")

    def arm(self):
        """Start one acquisition and wait for the external trigger."""
        time_indisposed_ms = ctypes.c_double(0)

        self.status["runBlock"] = ps.ps6000aRunBlock(
            self.chandle,
            PRE_TRIGGER,
            POST_TRIGGER,
            self.timebase,
            ctypes.byref(time_indisposed_ms),
            0,      # segment index
            None,   # no callback
            None
        )
        assert_pico_ok(self.status["runBlock"])

    def read_trace_mv(self):
        """Retrieve a triggered waveform in millivolts."""
        ready = ctypes.c_int16(0)
        t0 = time.time()

        while ready.value == 0:
            self.status["isReady"] = ps.ps6000aIsReady(
                self.chandle,
                ctypes.byref(ready)
            )

            if time.time() - t0 > TIMEOUT_S:
                raise TimeoutError(
                    "Timeout: PicoScope did not see trigger on Channel C."
                )

        no_of_samples = ctypes.c_uint64(N_SAMPLES)
        overflow = ctypes.c_int16(0)

        self.status["getValues"] = ps.ps6000aGetValues(
            self.chandle,
            0,
            ctypes.byref(no_of_samples),
            1,
            self.downsample_mode,
            0,
            ctypes.byref(overflow)
        )
        assert_pico_ok(self.status["getValues"])

        trace_mv = np.array(
            adc2mV(self.buffer_a_max, CHANNEL_A_RANGE, self.max_adc),
            dtype=np.float32
        )

        return trace_mv[:no_of_samples.value]

    def close(self):
        """Stop acquisitions and release the scope."""
        try:
            ps.ps6000aStop(self.chandle)
        except Exception:
            pass

        if self.chandle.value != 0:
            self.status["closeunit"] = ps.ps6000aCloseUnit(self.chandle)
            assert_pico_ok(self.status["closeunit"])
            print("PicoScope closed.")

In [ ]:
def random_block(size=16):
    """Generate a random AES plaintext block (16 bytes by default)."""
    return bytearray(np.random.randint(0, 256, size, dtype=np.uint8).tolist())


def run_target_once(target, plaintext, key):
    """Send one AES block to CW305 and read its 16-byte ciphertext."""
    target.set_key(key)
    target.simpleserial_write('p', plaintext)  # starts AES and creates trigger
    return target.simpleserial_read('r', 16)


def capture_once(target, pico, plaintext):
    """Arm the scope BEFORE triggering AES; return ciphertext and waveform."""
    pico.arm()
    time.sleep(0.002)                # same settling delay
    ciphertext = run_target_once(target, plaintext, FIXED_KEY)
    trace_mv = pico.read_trace_mv()

    # Reject incomplete captures instead of silently saving malformed rows.
    if ciphertext is None or len(ciphertext) != 16:
        raise RuntimeError("CW305 did not return a complete 16-byte ciphertext.")
    if len(trace_mv) != N_SAMPLES:
        raise RuntimeError(f"Expected {N_SAMPLES} samples, got {len(trace_mv)}.")
    return bytes(ciphertext), trace_mv

In [ ]:
# Catch an incorrect bitstream path before attempting a hardware connection.
for required_file in (BITSTREAM_PATH, DEFINES_PATH):
    if not Path(required_file).is_file():
        raise FileNotFoundError(f"CW305 setup file not found: {required_file}")

target = cw.target(
    None,
    cw.targets.CW305,
    bsfile=BITSTREAM_PATH,
    force=True,
    fpga_id=FPGA_ID,
    defines_files=[DEFINES_PATH],
)
print("CW305 connected. FPGA programmed:", target.is_programmed())

# Match the original CW305 setup.
target.vccint_set(1.0)
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)
target.clkusbautooff = True
target.clksleeptime = 1

pico = Pico6000EBlock()
pico.open()
pico.configure()
print("Hardware ready.")

In [ ]:
test_plaintext = random_block()
test_ciphertext, test_trace = capture_once(target, pico, test_plaintext)

print("Plaintext :", test_plaintext.hex())
print("Ciphertext:", test_ciphertext.hex())
print("Samples   :", len(test_trace))
print(f"Voltage   : {test_trace.min():.2f} to {test_trace.max():.2f} mV")

plt.figure(figsize=(11, 4))
plt.plot(test_trace, linewidth=0.9)
plt.title("Test AES trace — PicoScope Channel A")
plt.xlabel("Sample index")
plt.ylabel("Voltage (mV)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for output in (OUT_RANDOM, OUT_FIXED):
    if Path(output).exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(f"{output} already exists. Change OVERWRITE_OUTPUT or the filename.")

header = ["plaintext", "ciphertext"] + [f"sample_{i}" for i in range(N_SAMPLES)]
start_time = time.perf_counter()

try:
    with open(OUT_RANDOM, "w", newline="") as random_file, \
         open(OUT_FIXED, "w", newline="") as fixed_file:
        random_writer = csv.writer(random_file)
        fixed_writer = csv.writer(fixed_file)
        random_writer.writerow(header)
        fixed_writer.writerow(header)

        for i in tqdm(range(N_TRACES), desc="Collecting AES traces", unit="trace"):
            is_fixed = (i % 2 == 1)
            plaintext = FIXED_PLAINTEXT if is_fixed else random_block()
            ciphertext, trace_mv = capture_once(target, pico, plaintext)

            row = [bytes(plaintext).hex(), ciphertext.hex()] + trace_mv.tolist()
            (fixed_writer if is_fixed else random_writer).writerow(row)
finally:
    # These calls are essential to release the connected hardware.
    pico.close()
    if hasattr(target, "dis"):
        target.dis()

print(f"Collection finished in {time.perf_counter() - start_time:.1f} seconds")
print(f"Random traces: {N_TRACES // 2} -> {OUT_RANDOM}")
print(f"Fixed traces : {N_TRACES // 2} -> {OUT_FIXED}")

In [ ]:
def average_trace(csv_path, chunk_size=500):
    """Compute the mean waveform without loading the entire CSV into RAM."""
    total = None
    count = 0
    for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
        samples = chunk.iloc[:, 2:].to_numpy(dtype=np.float32)
        if total is None:
            total = np.zeros(samples.shape[1], dtype=np.float64)
        total += samples.sum(axis=0, dtype=np.float64)
        count += len(samples)
    if count == 0:
        raise ValueError(f"No data rows in {csv_path}")
    return total / count


avg_random = average_trace(OUT_RANDOM)
avg_fixed = average_trace(OUT_FIXED)

plt.figure(figsize=(11, 4))
plt.plot(avg_random, label="Random plaintexts")
plt.plot(avg_fixed, label="Fixed plaintext")
plt.title("Average AES waveforms")
plt.xlabel("Sample index")
plt.ylabel("Mean voltage (mV)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()